# 🩺 Inferensi Risiko Diabetes + Sistem Rekomendasi
## Model XGBoost — Prediksi Risiko Prediabetes pada Usia Muda
---
### 📋 Ringkasan Notebook

| Item | Keterangan |
|---|---|
| **Model** | XGBoost (`model_xgboost_risk_level.pkl`) |
| **Jumlah Fitur** | 12 fitur |
| **Output Label** | Tidak Berisiko / Sedang / Tinggi |
| **Persentase Risiko** | Weighted score dari `predict_proba` |
| **Rekomendasi** | Diurutkan berdasarkan feature importance model |

### 📌 Perubahan Input dari Versi Sebelumnya

| Fitur | Sebelumnya | Sekarang |
|---|---|---|
| `BMI` | Input angka langsung | Dihitung dari BB (kg) + TB (cm) |
| `Genetic_Risk_Score` | Angka 1–10 | 3 pilihan deskriptif |
| `Sleep_Hours` | Angka jam | 4 pilihan durasi |
| `Stress_Level` | Angka 1–10 | 3 pilihan deskriptif |

## STEP 1 — Import Library

In [11]:
import pandas as pd
import numpy as np
import joblib


print('✅ Library berhasil diimport!')

✅ Library berhasil diimport!


## STEP 2 — Load Model

> ⚠️ Pastikan file `model_xgboost_risk_level.pkl` sudah diupload ke Colab sebelum menjalankan cell ini.

In [12]:
# Load model yang sudah disimpan
# model_risk = joblib.load('model_xgboost_risk_level.pkl')
model_path = "../models"
model_risk = joblib.load(f"{model_path}/model_xgboost_risk_level.pkl")

# Ambil feature importance secara dinamis dari model
feature_names = [
    'Age', 'BMI', 'HbA1c', 'Fasting_Blood_Sugar',
    'Genetic_Risk_Score', 'Family_History_Diabetes',
    'Physical_Activity_Level', 'Dietary_Habits',
    'Smoking', 'Alcohol_Consumption', 'Sleep_Hours', 'Stress_Level'
]

importance_dict = dict(zip(feature_names, model_risk.feature_importances_))

label_map = {0: 'Tidak Berisiko', 1: 'Sedang', 2: 'Tinggi'}

print('✅ Model berhasil di-load!')
print()
print('Feature Importance (diurutkan):')
for feat, imp in sorted(importance_dict.items(), key=lambda x: x[1], reverse=True):
    bar = '█' * int(imp * 100)
    print(f'  {feat:<25}: {imp:.4f}  {bar}')

✅ Model berhasil di-load!

Feature Importance (diurutkan):
  HbA1c                    : 0.1736  █████████████████
  BMI                      : 0.1474  ██████████████
  Fasting_Blood_Sugar      : 0.1247  ████████████
  Family_History_Diabetes  : 0.0917  █████████
  Physical_Activity_Level  : 0.0737  ███████
  Dietary_Habits           : 0.0710  ███████
  Genetic_Risk_Score       : 0.0675  ██████
  Smoking                  : 0.0614  ██████
  Age                      : 0.0585  █████
  Alcohol_Consumption      : 0.0555  █████
  Stress_Level             : 0.0534  █████
  Sleep_Hours              : 0.0216  ██


## STEP 3 — Fungsi Kalkulasi & Mapping Input

Tiga fungsi berikut mengubah input user yang lebih mudah dipahami menjadi nilai numerik yang dibutuhkan model.

| Fungsi | Kegunaan |
|---|---|
| `hitung_bmi()` | Menghitung BMI dari berat & tinggi badan |
| `mapping_genetic()` | Mengubah pilihan riwayat keluarga ke skor genetik |
| `mapping_sleep()` | Mengubah pilihan durasi tidur ke angka jam |
| `mapping_stress()` | Mengubah pilihan tingkat stres ke angka 1–10 |

In [13]:
def hitung_bmi(berat_kg, tinggi_cm):
    """
    Menghitung BMI dari berat badan (kg) dan tinggi badan (cm).
    Rumus: BMI = BB / (TB dalam meter)^2
    """
    tinggi_m = tinggi_cm / 100
    bmi = berat_kg / (tinggi_m ** 2)

    if bmi < 18.5:
        kategori = 'Underweight'
    elif bmi < 25:
        kategori = 'Normal'
    elif bmi < 30:
        kategori = 'Overweight'
    else:
        kategori = 'Obesitas'

    return round(bmi, 2), kategori


def mapping_genetic(pilihan):
    """
    Mengubah pilihan riwayat keluarga menjadi Genetic Risk Score (1–10).
    Pilihan:
      - 'Tidak ada keluarga diabetes'     → 2
      - 'Ada (paman/bibi/kakek/nenek)'    → 5
      - 'Ada (ayah/ibu/saudara kandung)'  → 9
    """
    mapping = {
        'Tidak ada keluarga diabetes':       2,
        'Ada (paman/bibi/kakek/nenek)':      5,
        'Ada (ayah/ibu/saudara kandung)':    9
    }
    return mapping[pilihan]


def mapping_sleep(pilihan):
    """
    Mengubah pilihan durasi tidur menjadi angka jam.
    Pilihan:
      - 'Kurang dari 5 jam'  → 4.5
      - '5-6 jam'            → 5.5
      - '7-8 jam (ideal)'    → 7.5
      - 'Lebih dari 8 jam'   → 9.0
    """
    mapping = {
        'Kurang dari 5 jam':   4.5,
        '5-6 jam':             5.5,
        '7-8 jam (ideal)':     7.5,
        'Lebih dari 8 jam':    9.0
    }
    return mapping[pilihan]


def mapping_stress(pilihan):
    """
    Mengubah pilihan tingkat stres menjadi angka (1–10).
    Pilihan:
      - 'Hampir tidak pernah stres'          → 2
      - 'Kadang-kadang stres'                → 5
      - 'Sering stres / sulit dikendalikan'  → 8
    """
    mapping = {
        'Hampir tidak pernah stres':          2,
        'Kadang-kadang stres':                5,
        'Sering stres / sulit dikendalikan':  8
    }
    return mapping[pilihan]


print('✅ Fungsi kalkulasi & mapping berhasil didefinisikan!')

✅ Fungsi kalkulasi & mapping berhasil didefinisikan!


## STEP 4 — Fungsi Sistem Rekomendasi

Rekomendasi dibangkitkan berdasarkan kondisi tiap fitur input user, kemudian **diurutkan dari yang paling berpengaruh ke model** berdasarkan `feature_importances_` XGBoost secara dinamis.

### Logika Persentase Risiko (Opsi B — Weighted Score)
```
Risk% = (prob_Sedang × 0.5 + prob_Tinggi × 1.0) × 100
```
- Kelas **Tinggi** → bobot penuh (1.0)
- Kelas **Sedang** → bobot setengah (0.5)
- Kelas **Tidak Berisiko** → tidak berkontribusi (0.0)

In [14]:
def generate_recommendations(data, risk_label, importance_dict):
    """
    Membangkitkan rekomendasi personal berdasarkan kondisi fitur input.
    Diurutkan dari fitur dengan importance tertinggi ke terendah.

    Args:
        data          : dict berisi nilai fitur user (sudah dalam bentuk numerik)
        risk_label    : label prediksi ('Tidak Berisiko', 'Sedang', 'Tinggi')
        importance_dict: dict {nama_fitur: importance_score} dari model

    Returns:
        list of str: rekomendasi yang sudah diurutkan
    """
    recs = []  # list of (teks_rekomendasi, importance_score)

    # ── HbA1c ────────────────────────────────────────────
    if data['HbA1c'] >= 6.5:
        recs.append((
            '🩸 HbA1c kamu ≥6.5%, ini indikasi diabetes. Segera konsultasikan ke dokter untuk evaluasi lebih lanjut.',
            importance_dict['HbA1c']
        ))
    elif data['HbA1c'] >= 5.7:
        recs.append((
            '🩸 HbA1c kamu di rentang prediabetes (5.7–6.4%). Kurangi konsumsi gula dan karbohidrat sederhana, pantau secara berkala.',
            importance_dict['HbA1c']
        ))

    # ── BMI ──────────────────────────────────────────────
    if data['BMI'] >= 30:
        recs.append((
            '⚖️ BMI kamu masuk kategori Obesitas. Targetkan penurunan berat badan bertahap dengan diet seimbang dan olahraga rutin. Konsultasikan ke ahli gizi.',
            importance_dict['BMI']
        ))
    elif data['BMI'] >= 25:
        recs.append((
            '⚖️ BMI kamu masuk kategori Overweight. Jaga pola makan dan mulai olahraga rutin untuk mencapai BMI ideal (18.5–24.9).',
            importance_dict['BMI']
        ))

    # ── Fasting Blood Sugar ───────────────────────────────
    if data['Fasting_Blood_Sugar'] >= 126:
        recs.append((
            '🍬 Gula darah puasa kamu ≥126 mg/dL, indikasi diabetes. Segera periksakan ke dokter.',
            importance_dict['Fasting_Blood_Sugar']
        ))
    elif data['Fasting_Blood_Sugar'] >= 100:
        recs.append((
            '🍬 Gula darah puasa kamu 100–125 mg/dL (prediabetes). Kurangi karbohidrat sederhana dan gula tambahan.',
            importance_dict['Fasting_Blood_Sugar']
        ))

    # ── Family History ────────────────────────────────────
    if data['Family_History_Diabetes'] == 1:
        recs.append((
            '👨\u200d👩\u200d👧 Kamu memiliki riwayat keluarga diabetes. Lakukan skrining gula darah rutin minimal 1x/tahun.',
            importance_dict['Family_History_Diabetes']
        ))

    # ── Physical Activity ─────────────────────────────────
    if data['Physical_Activity_Level'] == 0:
        recs.append((
            '🏃 Kamu tergolong sedentary. Mulai lakukan minimal 150 menit aktivitas aerobik sedang per minggu (jalan cepat, bersepeda, berenang).',
            importance_dict['Physical_Activity_Level']
        ))
    elif data['Physical_Activity_Level'] == 1:
        recs.append((
            '🚶 Aktivitas fisik kamu cukup tapi belum optimal. Tingkatkan ke level aktif dengan target 150–300 menit/minggu.',
            importance_dict['Physical_Activity_Level']
        ))

    # ── Genetic Risk Score ────────────────────────────────
    if data['Genetic_Risk_Score'] >= 8:
        recs.append((
            '🧬 Risiko genetik kamu tinggi. Perhatikan lebih ketat pola makan dan gaya hidup sebagai bentuk pencegahan dini.',
            importance_dict['Genetic_Risk_Score']
        ))
    elif data['Genetic_Risk_Score'] >= 5:
        recs.append((
            '🧬 Risiko genetik kamu sedang. Tetap jaga gaya hidup sehat untuk meminimalkan risiko.',
            importance_dict['Genetic_Risk_Score']
        ))

    # ── Dietary Habits ────────────────────────────────────
    if data['Dietary_Habits'] == 0:
        recs.append((
            '🥗 Pola makan kamu tidak sehat. Terapkan pola makan seimbang: perbanyak sayur, buah, biji-bijian, dan kurangi lemak jenuh.',
            importance_dict['Dietary_Habits']
        ))
    elif data['Dietary_Habits'] == 1:
        recs.append((
            '🥦 Pola makan kamu cukup, tapi bisa lebih baik. Konsistenkan asupan bergizi setiap hari.',
            importance_dict['Dietary_Habits']
        ))

    # ── Stress Level ──────────────────────────────────────
    if data['Stress_Level'] >= 7:
        recs.append((
            '🧘 Tingkat stres kamu tinggi. Stres kronis menghambat sensitivitas insulin. Coba meditasi, yoga, atau konseling.',
            importance_dict['Stress_Level']
        ))
    elif data['Stress_Level'] >= 4:
        recs.append((
            '😌 Stres kamu di level sedang. Kelola dengan aktivitas relaksasi seperti jalan santai atau hobi.',
            importance_dict['Stress_Level']
        ))

    # ── Sleep Hours ───────────────────────────────────────
    if data['Sleep_Hours'] < 6:
        recs.append((
            '😴 Durasi tidur kamu kurang dari 6 jam. Kurang tidur meningkatkan risiko diabetes 28%. Targetkan 7–9 jam/malam.',
            importance_dict['Sleep_Hours']
        ))

    # ── Smoking ───────────────────────────────────────────
    if data['Smoking'] == 1:
        recs.append((
            '🚭 Kamu merokok. Merokok meningkatkan risiko diabetes tipe 2 sebesar 30–40%. Sangat disarankan untuk berhenti.',
            importance_dict['Smoking']
        ))

    # ── Alcohol ───────────────────────────────────────────
    if data['Alcohol_Consumption'] == 1:
        recs.append((
            '🍺 Kamu mengonsumsi alkohol. Alkohol berlebihan mengganggu metabolisme glukosa. Batasi atau hentikan.',
            importance_dict['Alcohol_Consumption']
        ))

    # Sort by importance (tertinggi duluan)
    recs.sort(key=lambda x: x[1], reverse=True)

    # Rekomendasi umum berdasarkan risk label (selalu di akhir)
    if risk_label == 'Tinggi':
        recs.append(('🏥 PRIORITAS: Segera konsultasikan kondisi kamu ke dokter atau ahli gizi untuk evaluasi menyeluruh.', 0))
    elif risk_label == 'Sedang':
        recs.append(('📋 Lakukan pemeriksaan kesehatan berkala setiap 6 bulan untuk memantau perkembangan kondisi kamu.', 0))
    else:
        recs.append(('✅ Pertahankan gaya hidup sehat kamu! Tetap lakukan pemeriksaan tahunan sebagai deteksi dini.', 0))

    return [r[0] for r in recs]


print('✅ Fungsi rekomendasi berhasil didefinisikan!')

✅ Fungsi rekomendasi berhasil didefinisikan!


## STEP 5 — Fungsi Utama Prediksi

Fungsi `prediksi_diabetes()` menerima input user, menjalankan kalkulasi & mapping, melakukan prediksi, dan menampilkan hasil lengkap beserta rekomendasi.

In [18]:
def prediksi_diabetes(
    usia,
    berat_kg,
    tinggi_cm,
    hba1c,
    fasting_blood_sugar,
    pilihan_genetic,
    family_history,
    physical_activity,
    dietary_habits,
    smoking,
    alcohol,
    pilihan_sleep,
    pilihan_stress
):
    """
    Fungsi utama prediksi risiko diabetes.

    Args:
        usia                : int   — usia user (tahun)
        berat_kg            : float — berat badan (kg)
        tinggi_cm           : float — tinggi badan (cm)
        hba1c               : float — nilai HbA1c (%)
        fasting_blood_sugar : float — gula darah puasa (mg/dL)
        pilihan_genetic     : str   — pilihan riwayat keluarga diabetes
        family_history      : int   — riwayat keluarga (1=Ya, 0=Tidak)
        physical_activity   : int   — 0=Sedentary, 1=Moderate, 2=Active
        dietary_habits      : int   — 0=Unhealthy, 1=Moderate, 2=Healthy
        smoking             : int   — 1=Ya, 0=Tidak
        alcohol             : int   — 1=Ya, 0=Tidak
        pilihan_sleep       : str   — pilihan durasi tidur
        pilihan_stress      : str   — pilihan tingkat stres

    Returns:
        dict berisi risk_label, risk_pct, bmi, proba, recommendations
    """
    # ── Kalkulasi & Mapping ───────────────────────────────
    bmi, kategori_bmi = hitung_bmi(berat_kg, tinggi_cm)
    genetic_risk      = mapping_genetic(pilihan_genetic)
    sleep_hours       = mapping_sleep(pilihan_sleep)
    stress_level      = mapping_stress(pilihan_stress)

    # ── Susun Data untuk Model ────────────────────────────
    data = {
    'Age':                      usia,
    'BMI':                      bmi,
    'HbA1c':                    hba1c,
    'Fasting_Blood_Sugar':      fasting_blood_sugar,
    'Genetic_Risk_Score':       genetic_risk,
    'Family_History_Diabetes':  family_history,
    'Physical_Activity_Level':  physical_activity,
    'Dietary_Habits':           dietary_habits,
    'Smoking':                  smoking,
    'Alcohol_Consumption':      alcohol,
    'Sleep_Hours':              sleep_hours,
    'Stress_Level':             stress_level
}

    # ── Prediksi ──────────────────────────────────────────
    df_input   = pd.DataFrame([data])
    pred       = model_risk.predict(df_input)[0]
    pred_proba = model_risk.predict_proba(df_input)[0]
    risk_label = label_map[pred]

    # ── Hitung Risk% (Weighted Score / Opsi B) ────────────
    # Risk% = (prob_Sedang × 0.5 + prob_Tinggi × 1.0) × 100
    risk_pct = (pred_proba[1] * 0.5 + pred_proba[2] * 1.0) * 100

    # ── Generate Rekomendasi ──────────────────────────────
    recommendations = generate_recommendations(data, risk_label, importance_dict)

    # ── Tampilkan Hasil ───────────────────────────────────
    print('=' * 60)
    print('          HASIL PREDIKSI RISIKO DIABETES')
    print('=' * 60)
    print(f'  Usia              : {usia} tahun')
    print(f'  Berat Badan       : {berat_kg} kg')
    print(f'  Tinggi Badan      : {tinggi_cm} cm')
    print(f'  BMI               : {bmi} ({kategori_bmi})')
    print(f'  HbA1c             : {hba1c}%')
    print(f'  Gula Darah Puasa  : {fasting_blood_sugar} mg/dL')
    print()
    print(f'  Risk Level        : {risk_label}')
    print(f'  Diabetes Risk     : {risk_pct:.1f}%')
    print()
    print('  Probabilitas per kelas:')
    for cls, name in label_map.items():
        bar = '█' * int(pred_proba[cls] * 30)
        print(f'    {name:<18}: {pred_proba[cls]:.2%}  {bar}')
    print()
    print('=' * 60)
    print('          REKOMENDASI PERSONAL')
    print('=' * 60)
    for i, rec in enumerate(recommendations, 1):
        print(f'  {i}. {rec}')
    print('=' * 60)

    return {
        'risk_label':      risk_label,
        'risk_pct':        round(risk_pct, 1),
        'bmi':             bmi,
        'kategori_bmi':    kategori_bmi,
        'proba': {
            'tidak_berisiko': round(float(pred_proba[0]), 4),
            'sedang':         round(float(pred_proba[1]), 4),
            'tinggi':         round(float(pred_proba[2]), 4)
        },
        'recommendations': recommendations
    }


print('✅ Fungsi prediksi berhasil didefinisikan!')

✅ Fungsi prediksi berhasil didefinisikan!


## STEP 6 — Panduan Nilai Input

Sebelum menjalankan prediksi, pastikan nilai input sesuai panduan berikut:

| Parameter | Tipe | Nilai yang Valid |
|---|---|---|
| `usia` | int | 15 – 25 tahun |
| `berat_kg` | float | Berat badan dalam kg |
| `tinggi_cm` | float | Tinggi badan dalam cm |
| `hba1c` | float | 4.0 – 10.0 (%) |
| `fasting_blood_sugar` | float | 70 – 180 (mg/dL) |
| `pilihan_genetic` | str | `'Tidak ada keluarga diabetes'` / `'Ada (paman/bibi/kakek/nenek)'` / `'Ada (ayah/ibu/saudara kandung)'` |
| `family_history` | int | `1` = Ya, `0` = Tidak |
| `physical_activity` | int | `0` = Sedentary, `1` = Moderate, `2` = Active |
| `dietary_habits` | int | `0` = Unhealthy, `1` = Moderate, `2` = Healthy |
| `smoking` | int | `1` = Ya, `0` = Tidak |
| `alcohol` | int | `1` = Ya, `0` = Tidak |
| `pilihan_sleep` | str | `'Kurang dari 5 jam'` / `'5-6 jam'` / `'7-8 jam (ideal)'` / `'Lebih dari 8 jam'` |
| `pilihan_stress` | str | `'Hampir tidak pernah stres'` / `'Kadang-kadang stres'` / `'Sering stres / sulit dikendalikan'` |

## STEP 7 — Jalankan Prediksi

Ganti nilai-nilai di bawah sesuai data yang ingin diprediksi.

In [19]:
# ══════════════════════════════════════════════════════════
# Ganti nilai di sini sesuai data user yang ingin diprediksi
# ══════════════════════════════════════════════════════════

hasil = prediksi_diabetes(
    usia                = 23,
    berat_kg            = 75,
    tinggi_cm           = 170,
    hba1c               = 5.9,
    fasting_blood_sugar = 105,
    pilihan_genetic     = 'Ada (ayah/ibu/saudara kandung)',
    family_history      = 1,
    physical_activity   = 1,      # 0=Sedentary, 1=Moderate, 2=Active
    dietary_habits      = 1,      # 0=Unhealthy, 1=Moderate, 2=Healthy
    smoking             = 0,
    alcohol             = 0,
    pilihan_sleep       = '7-8 jam (ideal)',
    pilihan_stress      = 'Kadang-kadang stres'
)

          HASIL PREDIKSI RISIKO DIABETES
  Usia              : 23 tahun
  Berat Badan       : 75 kg
  Tinggi Badan      : 170 cm
  BMI               : 25.95 (Overweight)
  HbA1c             : 5.9%
  Gula Darah Puasa  : 105 mg/dL

  Risk Level        : Tinggi
  Diabetes Risk     : 86.1%

  Probabilitas per kelas:
    Tidak Berisiko    : 0.36%  
    Sedang            : 27.13%  ████████
    Tinggi            : 72.50%  █████████████████████

          REKOMENDASI PERSONAL
  1. 🩸 HbA1c kamu di rentang prediabetes (5.7–6.4%). Kurangi konsumsi gula dan karbohidrat sederhana, pantau secara berkala.
  2. ⚖️ BMI kamu masuk kategori Overweight. Jaga pola makan dan mulai olahraga rutin untuk mencapai BMI ideal (18.5–24.9).
  3. 🍬 Gula darah puasa kamu 100–125 mg/dL (prediabetes). Kurangi karbohidrat sederhana dan gula tambahan.
  4. 👨‍👩‍👧 Kamu memiliki riwayat keluarga diabetes. Lakukan skrining gula darah rutin minimal 1x/tahun.
  5. 🚶 Aktivitas fisik kamu cukup tapi belum optimal. Tingkatkan ke 

## STEP 8 — Test dengan Beberapa Kasus Sekaligus

Cell ini menjalankan 4 skenario berbeda untuk memverifikasi bahwa sistem rekomendasi dan persentase risiko bekerja dengan benar.

In [21]:
test_cases = [
    {
        'nama': 'TEST 1 — Remaja sehat, semua nilai aman',
        'params': dict(
            usia=18, berat_kg=55, tinggi_cm=168,
            hba1c=4.5, fasting_blood_sugar=80,
            pilihan_genetic='Tidak ada keluarga diabetes',
            family_history=0, physical_activity=2, dietary_habits=2,
            smoking=0, alcohol=0,
            pilihan_sleep='7-8 jam (ideal)',
            pilihan_stress='Hampir tidak pernah stres'
        )
    },
    {
        'nama': 'TEST 2 — Dewasa muda, gaya hidup kurang baik',
        'params': dict(
            usia=23, berat_kg=80, tinggi_cm=170,
            hba1c=5.9, fasting_blood_sugar=105,
            pilihan_genetic='Ada (ayah/ibu/saudara kandung)',
            family_history=1, physical_activity=1, dietary_habits=1,
            smoking=0, alcohol=0,
            pilihan_sleep='7-8 jam (ideal)',
            pilihan_stress='Kadang-kadang stres'
        )
    },
    {
        'nama': 'TEST 3 — Semua faktor risiko tinggi',
        'params': dict(
            usia=25, berat_kg=100, tinggi_cm=165,
            hba1c=7.0, fasting_blood_sugar=140,
            pilihan_genetic='Ada (ayah/ibu/saudara kandung)',
            family_history=1, physical_activity=0, dietary_habits=0,
            smoking=1, alcohol=1,
            pilihan_sleep='Kurang dari 5 jam',
            pilihan_stress='Sering stres / sulit dikendalikan'
        )
    },
    {
        'nama': 'TEST 4 — Genetik tinggi tapi gaya hidup sehat',
        'params': dict(
            usia=22, berat_kg=60, tinggi_cm=165,
            hba1c=5.2, fasting_blood_sugar=90,
            pilihan_genetic='Ada (ayah/ibu/saudara kandung)',
            family_history=1, physical_activity=2, dietary_habits=2,
            smoking=0, alcohol=0,
            pilihan_sleep='7-8 jam (ideal)',
            pilihan_stress='Hampir tidak pernah stres'
        )
    }
]

# Jalankan semua test case
semua_hasil = []
for tc in test_cases:
    print(f"\n{'='*60}")
    print(f"  {tc['nama']}")
    print(f"{'='*60}")
    hasil = prediksi_diabetes(**tc['params'])
    semua_hasil.append((tc['nama'], hasil))

# Ringkasan
print(f"\n{'='*60}")
print('  RINGKASAN SEMUA TEST')
print(f"{'='*60}")
print(f"  {'Kasus':<35} {'Risk Level':<18} {'Risk%':<10} {'BMI'}")
print(f"  {'-'*35} {'-'*18} {'-'*10} {'-'*10}")
for nama, h in semua_hasil:
    label = nama.split('—')[1].strip() if '—' in nama else nama
    print(f"  {label:<35} {h['risk_label']:<18} {h['risk_pct']:<10.1f} {h['bmi']} ({h['kategori_bmi']})")
print(f"{'='*60}")


  TEST 1 — Remaja sehat, semua nilai aman
          HASIL PREDIKSI RISIKO DIABETES
  Usia              : 18 tahun
  Berat Badan       : 55 kg
  Tinggi Badan      : 168 cm
  BMI               : 19.49 (Normal)
  HbA1c             : 4.5%
  Gula Darah Puasa  : 80 mg/dL

  Risk Level        : Tidak Berisiko
  Diabetes Risk     : 0.0%

  Probabilitas per kelas:
    Tidak Berisiko    : 99.98%  █████████████████████████████
    Sedang            : 0.01%  
    Tinggi            : 0.00%  

          REKOMENDASI PERSONAL
  1. ✅ Pertahankan gaya hidup sehat kamu! Tetap lakukan pemeriksaan tahunan sebagai deteksi dini.

  TEST 2 — Dewasa muda, gaya hidup kurang baik
          HASIL PREDIKSI RISIKO DIABETES
  Usia              : 23 tahun
  Berat Badan       : 80 kg
  Tinggi Badan      : 170 cm
  BMI               : 27.68 (Overweight)
  HbA1c             : 5.9%
  Gula Darah Puasa  : 105 mg/dL

  Risk Level        : Tinggi
  Diabetes Risk     : 86.1%

  Probabilitas per kelas:
    Tidak Berisiko    :